#### Config Spark

In [1]:
import sys
sys.path.insert(0, '/opt/spark/python')
sys.path.insert(0, '/opt/spark/python/lib/py4j-0.10.9.7-src.zip')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ExploreData") \
    .config("spark.master", "spark://spark-master:7077") \
    .getOrCreate()

print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/12 18:43:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


#### Data Sources

In [2]:
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/application_train.csv", header=True, inferSchema=True).createOrReplaceTempView("applicationTrain")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/application_test.csv", header=True, inferSchema=True).createOrReplaceTempView("applicationTest")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/POS_CASH_balance.csv", header=True, inferSchema=True).createOrReplaceTempView("posCashBalance")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/bureau.csv", header=True, inferSchema=True).createOrReplaceTempView("bureau")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/bureau_balance.csv", header=True, inferSchema=True).createOrReplaceTempView("bureauBalance")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/credit_card_balance.csv", header=True, inferSchema=True).createOrReplaceTempView("creditCardBalance")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/installments_payments.csv", header=True, inferSchema=True).createOrReplaceTempView("installmentsPayments")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/previous_application.csv", header=True, inferSchema=True).createOrReplaceTempView("previousApplication")
spark.read.csv("hdfs://namenode:8020/data/dev/bronze/home_credit/raw/HomeCredit_columns_description.csv", header=True, inferSchema=True).createOrReplaceTempView("columnDescription")

26/08/12 18:43:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

#### applicationRaw

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

applications = spark.sql("""
SELECT 
    SK_ID_CURR AS loanId,
    TARGET AS target,
    NAME_CONTRACT_TYPE AS contractType,
    CODE_GENDER AS gender,
    FLAG_OWN_CAR AS ownCar,
    FLAG_OWN_REALTY AS ownRealty,
    CNT_CHILDREN AS childrenCnt,
    AMT_INCOME_TOTAL AS incomeTotal,
    AMT_CREDIT AS creditAmt,
    AMT_ANNUITY AS annuityAmt,
    AMT_GOODS_PRICE AS goodsPrice,
    NAME_TYPE_SUITE AS suiteType,
    NAME_INCOME_TYPE AS incomeType,
    NAME_EDUCATION_TYPE AS education,
    NAME_FAMILY_STATUS AS familyStatus,
    NAME_HOUSING_TYPE AS housingType,
    REGION_POPULATION_RELATIVE AS regionPop,
    FLOOR(ABS(DAYS_BIRTH) / 365.25) AS ageYears,
    CASE 
        WHEN DAYS_EMPLOYED = 365243 THEN 0
        ELSE ROUND(ABS(DAYS_EMPLOYED) / 365.25, 1)   -- ← ubah FLOOR menjadi ROUND
    END AS yearsEmployed,
    CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END AS isUnemployed,    
    DAYS_REGISTRATION AS daysReg,
    DAYS_ID_PUBLISH AS daysIdPub,
    OWN_CAR_AGE AS carAge,
    FLAG_MOBIL AS flagMobil,
    FLAG_EMP_PHONE AS empPhone,
    FLAG_WORK_PHONE AS workPhone,
    FLAG_CONT_MOBILE AS contMobile,
    FLAG_PHONE AS flagPhone,
    FLAG_EMAIL AS flagEmail,
    OCCUPATION_TYPE AS occupation,
    CNT_FAM_MEMBERS AS famMembers,
    REGION_RATING_CLIENT AS regionRate,
    REGION_RATING_CLIENT_W_CITY AS regionRateCity,
    WEEKDAY_APPR_PROCESS_START AS weekdayApp,
    HOUR_APPR_PROCESS_START AS hourApp,
    REG_REGION_NOT_LIVE_REGION AS regNotLive,
    REG_REGION_NOT_WORK_REGION AS regNotWork,
    LIVE_REGION_NOT_WORK_REGION AS liveNotWork,
    REG_CITY_NOT_LIVE_CITY AS cityNotLive,
    REG_CITY_NOT_WORK_CITY AS cityNotWork,
    LIVE_CITY_NOT_WORK_CITY AS liveCityNotWork,
    ORGANIZATION_TYPE AS orgType,
    EXT_SOURCE_1 AS ext1,
    EXT_SOURCE_2 AS ext2,
    EXT_SOURCE_3 AS ext3,
    APARTMENTS_AVG AS avgApt,
    BASEMENTAREA_AVG AS avgBsmt,
    YEARS_BEGINEXPLUATATION_AVG AS avgYrsExpl,
    YEARS_BUILD_AVG AS avgYrsBuild,
    COMMONAREA_AVG AS avgCommArea,
    ELEVATORS_AVG AS avgElev,
    ENTRANCES_AVG AS avgEnt,
    FLOORSMAX_AVG AS avgFlrMax,
    FLOORSMIN_AVG AS avgFlrMin,
    LANDAREA_AVG AS avgLand,
    LIVINGAPARTMENTS_AVG AS avgLivApt,
    LIVINGAREA_AVG AS avgLivArea,
    NONLIVINGAPARTMENTS_AVG AS avgNonLivApt,
    NONLIVINGAREA_AVG AS avgNonLivArea,
    APARTMENTS_MODE AS modeApt,
    BASEMENTAREA_MODE AS modeBsmt,
    YEARS_BEGINEXPLUATATION_MODE AS modeYrsExpl,
    YEARS_BUILD_MODE AS modeYrsBuild,
    COMMONAREA_MODE AS modeCommArea,
    ELEVATORS_MODE AS modeElev,
    ENTRANCES_MODE AS modeEnt,
    FLOORSMAX_MODE AS modeFlrMax,
    FLOORSMIN_MODE AS modeFlrMin,
    LANDAREA_MODE AS modeLand,
    LIVINGAPARTMENTS_MODE AS modeLivApt,
    LIVINGAREA_MODE AS modeLivArea,
    NONLIVINGAPARTMENTS_MODE AS modeNonLivApt,
    NONLIVINGAREA_MODE AS modeNonLivArea,
    APARTMENTS_MEDI AS medApt,
    BASEMENTAREA_MEDI AS medBsmt,
    YEARS_BEGINEXPLUATATION_MEDI AS medYrsExpl,
    YEARS_BUILD_MEDI AS medYrsBuild,
    COMMONAREA_MEDI AS medCommArea,
    ELEVATORS_MEDI AS medElev,
    ENTRANCES_MEDI AS medEnt,
    FLOORSMAX_MEDI AS medFlrMax,
    FLOORSMIN_MEDI AS medFlrMin,
    LANDAREA_MEDI AS medLand,
    LIVINGAPARTMENTS_MEDI AS medLivApt,
    LIVINGAREA_MEDI AS medLivArea,
    NONLIVINGAPARTMENTS_MEDI AS medNonLivApt,
    NONLIVINGAREA_MEDI AS medNonLivArea,
    FONDKAPREMONT_MODE AS modeFond,
    HOUSETYPE_MODE AS modeHouse,
    TOTALAREA_MODE AS modeTotal,
    WALLSMATERIAL_MODE AS modeWall,
    EMERGENCYSTATE_MODE AS modeEmerg,
    OBS_30_CNT_SOCIAL_CIRCLE AS obs30,
    DEF_30_CNT_SOCIAL_CIRCLE AS def30,
    OBS_60_CNT_SOCIAL_CIRCLE AS obs60,
    DEF_60_CNT_SOCIAL_CIRCLE AS def60,
    DAYS_LAST_PHONE_CHANGE AS daysPhone,
    FLAG_DOCUMENT_2 AS doc2,
    FLAG_DOCUMENT_3 AS doc3,
    FLAG_DOCUMENT_4 AS doc4,
    FLAG_DOCUMENT_5 AS doc5,
    FLAG_DOCUMENT_6 AS doc6,
    FLAG_DOCUMENT_7 AS doc7,
    FLAG_DOCUMENT_8 AS doc8,
    FLAG_DOCUMENT_9 AS doc9,
    FLAG_DOCUMENT_10 AS doc10,
    FLAG_DOCUMENT_11 AS doc11,
    FLAG_DOCUMENT_12 AS doc12,
    FLAG_DOCUMENT_13 AS doc13,
    FLAG_DOCUMENT_14 AS doc14,
    FLAG_DOCUMENT_15 AS doc15,
    FLAG_DOCUMENT_16 AS doc16,
    FLAG_DOCUMENT_17 AS doc17,
    FLAG_DOCUMENT_18 AS doc18,
    FLAG_DOCUMENT_19 AS doc19,
    FLAG_DOCUMENT_20 AS doc20,
    FLAG_DOCUMENT_21 AS doc21,
    AMT_REQ_CREDIT_BUREAU_HOUR AS reqBurH,
    AMT_REQ_CREDIT_BUREAU_DAY AS reqBurD,
    AMT_REQ_CREDIT_BUREAU_WEEK AS reqBurW,
    AMT_REQ_CREDIT_BUREAU_MON AS reqBurM,
    AMT_REQ_CREDIT_BUREAU_QRT AS reqBurQ,
    AMT_REQ_CREDIT_BUREAU_YEAR AS reqBurY
FROM applicationTrain
""")

applications.createOrReplaceTempView("application")

In [4]:
preview = spark.sql("""
    SELECT *
    FROM application
    LIMIT 5
""")
preview.toPandas()

,loanId,target,contractType,gender,ownCar,ownRealty,childrenCnt,incomeTotal,creditAmt,annuityAmt,goodsPrice,suiteType,incomeType,education,familyStatus,housingType,regionPop,ageYears,yearsEmployed,isUnemployed,daysReg,daysIdPub,carAge,flagMobil,empPhone,workPhone,contMobile,flagPhone,flagEmail,occupation,famMembers,regionRate,regionRateCity,weekdayApp,hourApp,regNotLive,regNotWork,liveNotWork,cityNotLive,cityNotWork,liveCityNotWork,orgType,ext1,ext2,ext3,avgApt,avgBsmt,avgYrsExpl,avgYrsBuild,avgCommArea,avgElev,avgEnt,avgFlrMax,avgFlrMin,avgLand,avgLivApt,avgLivArea,avgNonLivApt,avgNonLivArea,modeApt,modeBsmt,modeYrsExpl,modeYrsBuild,modeCommArea,modeElev,modeEnt,modeFlrMax,modeFlrMin,modeLand,modeLivApt,modeLivArea,modeNonLivApt,modeNonLivArea,medApt,medBsmt,medYrsExpl,medYrsBuild,medCommArea,medElev,medEnt,medFlrMax,medFlrMin,medLand,medLivApt,medLivArea,medNonLivApt,medNonLivArea,modeFond,modeHouse,modeTotal,modeWall,modeEmerg,obs30,def30,obs60,def60,daysPhone,doc2,doc3,doc4,doc5,doc6,doc7,doc8,doc9,doc10,doc11,doc12,doc13,doc14,doc15,doc16,doc17,doc18,doc19,doc20,doc21,reqBurH,reqBurD,reqBurW,reqBurM,reqBurQ,reqBurY
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,25,1.7,0,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,45,3.3,0,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,52,0.6,0,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,None,None,0.0,0.0,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,52,8.3,0,-9833.0,-2437,NaN,1,1,0,1,0,0,Laborers,2.0,2,2,WEDNESDAY,17,0,0,0,0,0,0,Business Entity Type 3,NaN,0.650442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,None,None,2.0,0.0,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.028663,54,8.3,0,-4311.0,-3458,NaN,1,1,0,1,0,0,Core staff,1.0,2,2,THURSDAY,11,0,0,0,0,1,1,Religion,NaN,0.322738,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

#### CHECK ANOMALY

In [ ]:
eda = spark.sql("""
    SELECT 
        COUNT(*) AS jumlahLoadId,
        COUNT(
            CASE 
                WHEN loanId IS NULL THEN 1
            END
        ) AS nullCount,
        COUNT(loanId)-COUNT(DISTINCT loanId) AS duplicateCount
    FROM application 
""")
eda.toPandas()

#### Contract Type Analysis

In [ ]:
contract_eda = spark.sql("""
SELECT 
    contractType, 
    COUNT(*) AS total_customer,
    AVG(target) AS default_rate
FROM application
GROUP BY contractType
ORDER BY default_rate DESC
""")
contract_eda.toPandas()

In [ ]:
cash_risk_factors = spark.sql("""
SELECT 
    contractType,
    AVG(creditAmt) AS avg_credit,
    AVG(annuityAmt) AS avg_annuity,
    AVG(incomeTotal) AS avg_income,
    AVG(creditAmt / incomeTotal) AS avg_debt_to_income
FROM application
GROUP BY contractType
""")
cash_risk_factors.toPandas()

#### SUITE ANALYSIS

In [ ]:
suite_analysis = spark.sql("""
SELECT 
    suiteType, 
    COUNT(*) AS total_customer,
    AVG(target) AS default_rate
FROM application
GROUP BY suiteType
ORDER BY default_rate DESC
""")
suite_analysis.toPandas()

#### Car & Realy Analysis

In [ ]:
car_house_analysis = spark.sql("""
SELECT 
    ownCar,
    ownRealty,
    COUNT(*) AS total_customer,
    AVG(target) AS default_rate
FROM application
GROUP BY ownCar, ownRealty
ORDER BY default_rate DESC
""")
car_house_analysis.toPandas()

In [ ]:
car_house_income = spark.sql("""
SELECT 
    ownCar,
    ownRealty,
    AVG(incomeTotal) AS avg_income,
    AVG(annuityAmt) AS avg_annuity,
    AVG(creditAmt) AS avg_credit,
    AVG(annuityAmt / incomeTotal) AS avg_payment_ratio
FROM application
GROUP BY ownCar, ownRealty
""")
car_house_income.toPandas()

#### !! : Analysis Result : Realty was not effected on applicants

In [ ]:
annuity_distribution = spark.sql("""
SELECT
    ownRealty,
    COUNT(*) AS totalCust,
    MIN(annuityAmt) AS minAnnuity,
    MAX(annuityAmt) AS maxAnnuity,
    AVG(annuityAmt) AS avgAnnuity,
    PERCENTILE(annuityAmt, 0.5) AS median_annuity,  -- Ganti percentile_approx
    SUM(CASE WHEN annuityAmt > 100000 THEN 1 ELSE 0 END) AS countGt100k,
    ROUND(SUM(CASE WHEN annuityAmt > 100000 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pctGt100k
FROM application
GROUP BY ownRealty
""")
annuity_distribution.toPandas()

#### AGE ANALYSIS To Car & Realy

In [ ]:
age_analysis = spark.sql("""
SELECT 
    ownCar,
    ownRealty,
    COUNT(*) AS totalCustomer,
    AVG(ABS(daysBirth) / 365.25) AS avgAge
FROM application
GROUP BY ownCar, ownRealty
ORDER BY ownCar, ownRealty
""")
age_analysis.toPandas()

In [ ]:
family_profile = spark.sql("""
SELECT 
    ownCar,
    ownRealty,
    AVG(childrenCnt) AS avg_children,
    AVG(famMembers) AS avg_family_members,
    AVG(ABS(daysBirth) / 365.25) AS avg_age
FROM application
GROUP BY ownCar, ownRealty
ORDER BY ownCar, ownRealty
""")
family_profile.toPandas()

#### AGE Analysis To Academic

#### GoodPrices Analysis

In [ ]:
zero_price = spark.sql("""
SELECT COUNT(*) AS applicant_barang
FROM application
WHERE goodsPrice IS NOT NULL
""")
zero_price.toPandas()

In [ ]:
zero_price = spark.sql("""
SELECT COUNT(*) AS applicant_nonBarang
FROM application
WHERE goodsPrice = 0 OR goodsPrice IS NULL
""")
zero_price.toPandas()

In [ ]:
diff_price = spark.sql("""
SELECT 
    loanId,
    creditAmt,
    goodsPrice,
    (creditAmt - goodsPrice) AS diff
FROM application
WHERE goodsPrice > 0
ORDER BY diff DESC
LIMIT 20
""")
diff_price.toPandas()

In [ ]:
diff_neg = spark.sql("""
SELECT 
    loanId,
    creditAmt,
    goodsPrice,
    (creditAmt - goodsPrice) AS diff
FROM application
WHERE goodsPrice > 0
ORDER BY diff ASC
LIMIT 20
""")
diff_neg.toPandas()

In [ ]:
diff_zero = spark.sql("""
SELECT 
    loanId,
    creditAmt,
    goodsPrice,
    (creditAmt - goodsPrice) AS diff
FROM application
WHERE goodsPrice > 0 AND creditAmt = goodsPrice
LIMIT 20
""")
diff_zero.toPandas()

In [ ]:
diff_combined = spark.sql("""
(SELECT 'Positif (Pinjaman > Harga)' AS case_type, loanId, creditAmt, goodsPrice, (creditAmt - goodsPrice) AS diff
 FROM application
 WHERE goodsPrice > 0
 ORDER BY diff DESC
 LIMIT 10)
UNION ALL
(SELECT 'Negatif (Uang Muka)' AS case_type, loanId, creditAmt, goodsPrice, (creditAmt - goodsPrice) AS diff
 FROM application
 WHERE goodsPrice > 0
 ORDER BY diff ASC
 LIMIT 10)
""")
diff_combined.toPandas()

In [ ]:
diff_combined = spark.sql("""
(SELECT 'Positif (Pinjaman > Harga)' AS case_type, 
        loanId, 
        creditAmt, 
        goodsPrice, 
        (creditAmt - goodsPrice) AS diff,
        AVG(target) AS avg_target
 FROM application
 WHERE goodsPrice > 0
 GROUP BY loanId, creditAmt, goodsPrice, (creditAmt - goodsPrice), target
 ORDER BY diff DESC
 LIMIT 10)
UNION ALL
(SELECT 'Negatif (Uang Muka)' AS case_type, 
        loanId, 
        creditAmt, 
        goodsPrice, 
        (creditAmt - goodsPrice) AS diff,
        AVG(target) AS avg_target
 FROM application
 WHERE goodsPrice > 0
 GROUP BY loanId, creditAmt, goodsPrice, (creditAmt - goodsPrice), target
 ORDER BY diff ASC
 LIMIT 10)
""")
diff_combined.toPandas()

In [ ]:
income_analysis = spark.sql("""
SELECT 
    incomeType,
    COUNT(*) AS total_customer,
    AVG(target) AS default_rate
FROM application
GROUP BY incomeType
ORDER BY default_rate DESC
""")
income_analysis.toPandas()

In [ ]:
income_cross = spark.sql("""
SELECT 
    incomeType,
    COUNT(*) AS total,
    AVG(target) AS default_rate,
    AVG(incomeTotal) AS avg_income,
    AVG(creditAmt) AS avg_credit,
    AVG(creditAmt / incomeTotal) AS avg_ratio
FROM application
GROUP BY incomeType
ORDER BY default_rate DESC
""")
income_cross.toPandas()

#### Analysis Perpubahan Alamat Rumah (Reg) dan Perubahan KTP (Id)

In [ ]:
identity_analysis = spark.sql("""
SELECT 
    CASE 
        WHEN daysReg >= -365 THEN 'Reg: < 1 tahun'
        WHEN daysReg >= -1095 THEN 'Reg: 1-3 tahun'
        WHEN daysReg >= -3650 THEN 'Reg: 3-10 tahun'
        ELSE 'Reg: > 10 tahun'
    END AS reg_duration,
    CASE 
        WHEN daysIdPub >= -365 THEN 'ID: < 1 tahun'
        WHEN daysIdPub >= -1095 THEN 'ID: 1-3 tahun'
        WHEN daysIdPub >= -3650 THEN 'ID: 3-10 tahun'
        ELSE 'ID: > 10 tahun'
    END AS id_duration,
    COUNT(*) AS total,
    AVG(target) AS default_rate
FROM application
WHERE daysReg IS NOT NULL AND daysIdPub IS NOT NULL
GROUP BY 1, 2
ORDER BY default_rate DESC
""")
identity_analysis.toPandas()

#### Analisis Informasi Private dengan Applicants

In [ ]:
contact_score = spark.sql("""
SELECT 
    (flagMobil + empPhone + workPhone + contMobile + flagPhone + flagEmail) AS contact_score,
    COUNT(*) AS total,
    AVG(target) AS default_rate
FROM application
GROUP BY contact_score
ORDER BY contact_score DESC
""")
contact_score.toPandas()

In [8]:
occupation_eda = spark.sql("""
SELECT 
    occupation,
    AVG(famMembers) AS avg_family_members,
    COUNT(*) AS total,
    AVG(target) AS default_rate
FROM application
GROUP BY occupation
ORDER BY default_rate DESC
""")
occupation_eda.toPandas()

,occupation,avg_family_members,total,default_rate
0,Low-skill Laborers,2.194935,2093,0.171524
1,Drivers,2.339193,18603,0.113261
2,Waiters/barmen staff,2.082344,1348,0.112760
3,Security staff,2.130933,6721,0.107424
4,Laborers,2.276664,55186,0.105788
5,Cooking staff,2.224184,5946,0.104440
6,Sales staff,2.237244,32102,0.096318
7,Cleaning staff,2.034601,4653,0.096067
8,Realty agents,2.221039,751,0.078562
9,Secretaries,2.271264,1305,0.070498


#### Analytics Occupation on Income with Defailt Rate

In [6]:
occupation_deep = spark.sql("""
SELECT 
    occupation,
    COUNT(*) AS total,
    AVG(target) AS default_rate,
    AVG(incomeTotal) AS avg_income,
    AVG(creditAmt) AS avg_credit,
    AVG(creditAmt / incomeTotal) AS avg_ratio
FROM application
GROUP BY occupation
ORDER BY default_rate DESC
""")
occupation_deep.toPandas()

,occupation,total,default_rate,avg_income,avg_credit,avg_ratio
0,Low-skill Laborers,2093,0.171524,133228.001911,458464.781653,3.662340
1,Drivers,18603,0.113261,187011.606413,612333.969037,3.509671
2,Waiters/barmen staff,1348,0.112760,144272.583828,491451.966988,3.706955
3,Security staff,6721,0.107424,149662.695953,557080.293334,4.052667
4,Laborers,55186,0.105788,166357.482525,570617.995597,3.748026
5,Cooking staff,5946,0.104440,138396.508176,539174.315843,4.227093
6,Sales staff,32102,0.096318,152302.874710,563258.492088,4.014021
7,Cleaning staff,4653,0.096067,130790.895551,510960.949710,4.271460
8,Realty agents,751,0.078562,195003.994674,655275.649134,3.579616
9,Secretaries,1305,0.070498,160541.662069,593520.413793,4.188763


In [5]:
occupation_feature = spark.sql("""
SELECT 
    loanId,
    CASE 
        WHEN occupation IN ('Low-skill Laborers', 'Drivers', 'Waiters/barmen staff', 
                            'Security staff', 'Laborers', 'Cooking staff', 'Cleaning staff') 
            THEN 'High Risk'
        WHEN occupation IN ('Sales staff', 'Realty agents', 'Secretaries', 
                            'Private service staff', 'Core staff', 'None') 
            THEN 'Medium Risk'
        ELSE 'Low Risk'  
    END AS occupation_risk_group
FROM application
""")
occupation_feature.toPandas()

,loanId,occupation_risk_group
0,100002,High Risk
1,100003,Medium Risk
2,100004,High Risk
3,100006,High Risk
4,100007,Medium Risk
...,...,...
307506,456251,Medium Risk
307507,456252,Low Risk
307508,456253,Low Risk
307509,456254,High Risk
